# 1. collections.Counter vs manual counting

In [1]:
import time
import collections

words = ["apple", "banana", "cherry"] * 300_000

start = time.time()
counts = {}
for w in words:
    counts[w] = counts.get(w, 0) + 1
print(f"Manual dict count:    {time.time() - start:.4f}s")

start = time.time()
counts = collections.Counter(words)
print(f"Counter():            {time.time() - start:.4f}s")

Manual dict count:    0.0496s
Counter():            0.0155s


# 2. collections.deque vs list for queue (popleft)

In [3]:
import time
import collections

words = ["apple", "banana", "cherry"] * 300_000

start = time.time()
counts = {}
for w in words:
    if w in counts:
        counts[w] = counts[w] + 1
    else:
        counts[w] = 1
print(f"Manual dict count:    {time.time() - start:.4f}s")

start = time.time()
counts = collections.Counter(words)
print(f"Counter():            {time.time() - start:.4f}s")

Manual dict count:    0.0698s
Counter():            0.0156s


# 3. Generator expression vs list comprehension (memory & speed for sum)

In [4]:
n = 100_000

start = time.time()
q = list(range(n))
while q:
    q.pop(0)  # O(n) each time
print(f"list.pop(0):          {time.time() - start:.4f}s")

start = time.time()
q = collections.deque(range(n))
while q:
    q.popleft()  # O(1)
print(f"deque.popleft():      {time.time() - start:.4f}s")

list.pop(0):          0.7455s
deque.popleft():      0.0035s


# 4.str.join() vs += string concatenation

In [7]:
LARGE_NUMBER = 1_000_000

v1 = [i * i for i in range(LARGE_NUMBER)]

start = time.time()
total = sum(v1)
print(f"List comprehension:   {time.time() - start:.4f}s  | result={total}")

v2 = (i * i for i in range(LARGE_NUMBER))
start = time.time()
total = sum(v2)
print(f"Generator expr:       {time.time() - start:.4f}s  | result={total}")

List comprehension:   0.0078s  | result=333332833333500000
Generator expr:       0.0351s  | result=333332833333500000


# 5. io.StringIO vs string concatenation for building large text

In [8]:
parts = ["word"] * 100_000

start = time.time()
result = ""
for p in parts:
    result += p
print(f"str +=:               {time.time() - start:.4f}s  | len={len(result)}")

start = time.time()
result = "".join(parts)
print(f"str.join():           {time.time() - start:.4f}s  | len={len(result)}")

str +=:               0.4564s  | len=400000
str.join():           0.0006s  | len=400000


# 6. chain(intertools) vs io.StringIO vs string concatenation for building large text

In [12]:
from itertools import chain

parts = ["word"] * 100_000

start = time.time()
result = ""
for p in parts:
    result += p
print(f"str +=:               {time.time() - start:.4f}s  | len={len(result)}")

start = time.time()
result = "".join(parts)

print(f"str.join():           {time.time() - start:.4f}s  | len={len(result)}")


start = time.time()
result = list(chain(parts))

print(f"chain:           {time.time() - start:.4f}s  | len={len(result)}")

print("result", result)

str +=:               0.4455s  | len=400000
str.join():           0.0006s  | len=400000
chain:           0.0003s  | len=100000
result ['word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', 'word', '

# 7. map() vs list comprehension for simple transformations

In [13]:
import io
n = 50_000

start = time.time()
result = ""
for i in range(n):
    result += f"line {i}\n"
print(f"String concat:        {time.time() - start:.4f}s  | lines={n}")

start = time.time()
buf = io.StringIO()
for i in range(n):
    buf.write(f"line {i}\n")
result = buf.getvalue()
print(f"io.StringIO:          {time.time() - start:.4f}s  | lines={n}")

String concat:        0.3100s  | lines=50000
io.StringIO:          0.0050s  | lines=50000


In [14]:
numbers = list(range(LARGE_NUMBER))

start = time.time()
result = [str(n) for n in numbers]
print(f"List comprehension:   {time.time() - start:.4f}s")

start = time.time()
result = list(map(str, numbers))
print(f"map(str, ...):        {time.time() - start:.4f}s")

List comprehension:   0.0508s
map(str, ...):        0.0434s


# 8. functools.lru_cache for memoization

In [15]:
import  functools 

def fib_slow(n):
    if n < 2:
        return n
    return fib_slow(n - 1) + fib_slow(n - 2)

@functools.lru_cache(maxsize=None)
def fib_fast(n):
    if n < 2:
        return n
    return fib_fast(n - 1) + fib_fast(n - 2)

start = time.time()
result = fib_slow(35)
print(f"fib no cache (35):    {time.time() - start:.4f}s  | result={result}")

start = time.time()
result = fib_fast(35)
print(f"fib lru_cache (35):   {time.time() - start:.4f}s  | result={result}")

fib no cache (35):    0.6720s  | result=9227465
fib lru_cache (35):   0.0000s  | result=9227465


# 9. Local variable lookup vs global/attribute lookup inside tight loops

In [ ]:

import math

def use_global_attr():
    result = 0
    for i in range(LARGE_NUMBER):
        result += math.floor(i * 1.5)
    return result

def use_local_ref():
    _floor = math.floor   # local reference
    result = 0
    for i in range(LARGE_NUMBER):
        result += _floor(i * 1.5)
    return result

start = time.time()
use_global_attr()
print(f"Global attr lookup:   {time.time() - start:.4f}s")

start = time.time()
use_local_ref()
print(f"Local ref lookup:     {time.time() - start:.4f}s")



Global attr lookup:   0.0670s
Local ref lookup:     0.0387s


# # 10. zip() vs index-based parallel iteration

In [ ]:


a = list(range(LARGE_NUMBER))
b = list(range(LARGE_NUMBER, 2 * LARGE_NUMBER))

start = time.time()
total = 0
for i in range(len(a)):
    total += a[i] + b[i]
print(f"Index iteration:      {time.time() - start:.4f}s")

start = time.time()
total = 0
for x, y in zip(a, b):
    total += x + y
print(f"zip() iteration:      {time.time() - start:.4f}s")


Index iteration:      0.0735s
zip() iteration:      0.0630s


# # 11. enumerate() vs manual counter

In [ ]:

items = list(range(LARGE_NUMBER))

start = time.time()
idx = 0
result = []
for item in items:
    result.append((idx, item))
    idx += 1
print(f"Manual counter:       {time.time() - start:.4f}s")

start = time.time()
result = list(enumerate(items))
print(f"enumerate():          {time.time() - start:.4f}s")

Manual counter:       0.5071s
enumerate():          0.0517s


# 12. any() / all() short-circuit vs full loop

In [19]:
numbers = list(range(LARGE_NUMBER))

start = time.time()
found = False
for n in numbers:
    if n > 500_000:
        found = True
        break
print(f"Manual loop+break:    {time.time() - start:.6f}s  | found={found}")

start = time.time()
found = any(n > 500_000 for n in numbers)
print(f"any() short-circuit:  {time.time() - start:.6f}s  | found={found}")

Manual loop+break:    0.011233s  | found=True
any() short-circuit:  0.009026s  | found=True


In [20]:
any([]), all([])

(False, True)

# 13. Set intersection vs nested loop for common elements

In [21]:
list_a = list(range(0, 100_000, 2))
list_b = list(range(0, 100_000, 3))

start = time.time()
common = [x for x in list_a if x in list_b]
print(f"Nested list check:    {time.time() - start:.4f}s  | count={len(common)}")

start = time.time()
common = list(set(list_a) & set(list_b))
print(f"Set intersection:     {time.time() - start:.4f}s  | count={len(common)}")


Nested list check:    6.5821s  | count=16667
Set intersection:     0.0011s  | count=16667


# 14. manual transform  vs sorted() with key=




In [22]:
import random
words = ["".join(random.choices("abcdefghij", k=8)) for _ in range(100_000)]

start = time.time()
result = sorted(words, key=str.lower)   # built-in method reference
print(f"sorted(key=str.lower):{time.time() - start:.4f}s")

start = time.time()
result = sorted(words, key=lambda w: w.lower())
print(f"sorted(key=lambda):   {time.time() - start:.4f}s")


sorted(key=str.lower):0.0407s
sorted(key=lambda):   0.0224s


# 15. lambda  vs operator module




In [23]:
import random
words = ["".join(random.choices("abcdefghij", k=8)) for _ in range(100_000)]

lower_ref = str.lower

start = time.time()
result = sorted(words, key=lower_ref)   # built-in method reference
print(f"sorted(key=str.lower):{time.time() - start:.4f}s")

start = time.time()
result = sorted(words, key=lambda w: w.lower())
print(f"sorted(key=lambda):   {time.time() - start:.4f}s")

sorted(key=str.lower):0.0218s
sorted(key=lambda):   0.0233s


# 16. dict.get() vs KeyError handling


In [4]:


import operator
import random
import time

LARGE_NUMBER = 1_000_000

pairs = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(LARGE_NUMBER)]

start = time.time()
result = sorted(pairs, key=lambda p: p[1])
print(f"lambda key:           {time.time() - start:.4f}s")

start = time.time()
result = sorted(pairs, key= operator.itemgetter(1))
print(f"itemgetter key:       {time.time() - start:.4f}s")

lambda key:           0.1433s
itemgetter key:       0.1258s


# 17. operator module vs lambda for simple operations



In [6]:


import operator
import random
import time

LARGE_NUMBER = 1_000_000

pairs = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(LARGE_NUMBER)]


start = time.time()
result = sorted(pairs, key= operator.itemgetter(1))
print(f"itemgetter key:       {time.time() - start:.4f}s")

start = time.time()
result = sorted(pairs, key=lambda p: p[1])
print(f"lambda key:           {time.time() - start:.4f}s")



itemgetter key:       1.7102s
lambda key:           0.1606s


In [5]:
import operator

pairs = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(LARGE_NUMBER * 10)]

start = time.time()
result = sorted(pairs, key=lambda p: p[1])
print(f"lambda key:           {time.time() - start:.4f}s")

start = time.time()
result = sorted(pairs, key= operator.itemgetter(1))
print(f"itemgetter key:       {time.time() - start:.4f}s")

lambda key:           1.5493s
itemgetter key:       1.3237s


# 18. dict comprehension vs dict(zip(...))





In [7]:


import operator
import random
import time

LARGE_NUMBER = 1_000_000

pairs1 = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(LARGE_NUMBER)]

pairs2 = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(LARGE_NUMBER)]


start = time.time()
result = sorted(pairs1, key= operator.itemgetter(1))
print(f"itemgetter key:       {time.time() - start:.4f}s")

start = time.time()
result = sorted(pairs2, key=lambda p: p[1])
print(f"lambda key:           {time.time() - start:.4f}s")

itemgetter key:       0.1379s
lambda key:           0.1587s


In [8]:


import operator
import random
import time

LARGE_NUMBER = 1_000_000 

pairs1 = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(LARGE_NUMBER * 10)]

pairs2 = [(random.randint(0, 1000), random.randint(0, 1000)) for _ in range(LARGE_NUMBER * 10)]


start = time.time()
result = sorted(pairs1, key= operator.itemgetter(1))
print(f"itemgetter key:       {time.time() - start:.4f}s")

start = time.time()
result = sorted(pairs2, key=lambda p: p[1])
print(f"lambda key:           {time.time() - start:.4f}s")

KeyboardInterrupt: 

In [9]:
# 16. dict.get() vs KeyError handling

data = {i: i * 2 for i in range(LARGE_NUMBER)}
keys_to_lookup = list(range(LARGE_NUMBER)) + list(range(LARGE_NUMBER, LARGE_NUMBER + 1000))

start = time.time()
results = []
for k in keys_to_lookup:
    try:
        results.append(data[k])
    except KeyError:
        results.append(None)
print(f"try/except KeyError:  {time.time() - start:.4f}s")

start = time.time()
results = []
for k in keys_to_lookup:
    results.append(data.get(k))
print(f"dict.get():           {time.time() - start:.4f}s")

try/except KeyError:  0.0464s
dict.get():           0.0500s


# 18. dict comprehension vs dict(zip(...))

In [ ]:


keys = list(range(LARGE_NUMBER))
values = list(range(LARGE_NUMBER, 2 * LARGE_NUMBER))

start = time.time()
d = {k: v for k, v in zip(keys, values)}
print(f"Dict comprehension:   {time.time() - start:.4f}s")

start = time.time()
d = dict(zip(keys, values))
print(f"dict(zip()):          {time.time() - start:.4f}s")

Dict comprehension:   0.0359s
dict(zip()):          0.0229s


# 19. functools.reduce vs explicit loop for folding

In [ ]:


import functools


numbers = list(range(1, 10_001))

start = time.time()
product = 1
for n in numbers:
    product *= n
print(f"Explicit loop:        {time.time() - start:.4f}s")

start = time.time()
product = functools.reduce(operator.mul, numbers, 1)
print(f"functools.reduce:     {time.time() - start:.4f}s")

Explicit loop:        0.0456s
functools.reduce:     0.0309s


# 20. Unpacking swap vs temp variable

In [ ]:


def swap_temp(a, b):
    temp = a
    a = b
    b = temp
    return a, b
def swap_unpack(a, b):
    a, b = b, a
    return a, b
n = 5_000_000
start = time.time()
x, y = 1, 2
for _ in range(n):
    x, y = swap_temp(x, y)
    
print(f"Temp variable swap:   {time.time() - start:.4f}s")
start = time.time()
x, y = 1, 2
for _ in range(n):
    x, y = swap_unpack(x, y)
print(f"Tuple unpack swap:    {time.time() - start:.4f}s")

Temp variable swap:   0.3459s
Tuple unpack swap:    0.3221s


In [14]:
# 21. Use `in` on dict keys vs list of keys 


registry = {i: f"item_{i}" for i in range(LARGE_NUMBER)}
keys_list = list(registry.keys())

start = time.time()
found = sum(1 for k in range(0, 2000, 2) if k in keys_list)
print(f"'in' on list:         {time.time() - start:.6f}s  | found={found}")

start = time.time()
found = sum(1 for k in range(0, 2000, 2) if k in registry)
print(f"'in' on dict:         {time.time() - start:.6f}s  | found={found}")

'in' on list:         0.004703s  | found=1000
'in' on dict:         0.000105s  | found=1000
